# Wake Vision Mobile Export

Pipeline para exportar o dataset de teste **Wake Vision** em formatos otimizados para dispositivos móveis:

1. **NPZ**: Compactado, fácil para carregar em Python
2. **PNG + CSV**: Imagens individuais com metadados (classes, tamanhos originais)
3. **TFLite**: Modelos quantizados (opcionalmente) para Android/Raspberry Pi

O dataset tem **~55K imagens de tamanhos variados** do teste de pessoa vs não-pessoa no Wake Vision.
Uma amostra estratificada de **500 imagens** é exportada para avaliar em dispositivos.

## Seção 0 — Setup e Imports

In [ ]:
from __future__ import annotations

import gc
import json
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

# Configurações
SEED = 42
GPU_ID = 1
SAMPLE_SIZE = 500  # Número de imagens para amostrar
IMG_SIZE = 224

DATA_DIR = Path("/home/saulo/edgeml/data")
EXPORT_DIR = Path("/home/saulo/edgeml/export_wakevision")
MODELS_DIR = Path("/home/saulo/edgeml/keras_models")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Export dir: {EXPORT_DIR}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")

# GPU
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
if gpus and GPU_ID is not None:
    chosen = gpus[GPU_ID]
    tf.config.set_visible_devices([chosen], "GPU")
    tf.config.experimental.set_memory_growth(chosen, True)
    print(f"✓ GPU {GPU_ID}: {chosen.name}")

## Seção 1 — Carregado Wake Vision Test Set

In [ ]:
# Carrega dataset completo
print("Carregando Wake Vision test set...")
ds_test = tfds.load(
    "wake_vision",
    split="test",
    data_dir=str(DATA_DIR),
    as_supervised=False,
    shuffle_files=False,
)

# Remove exemplos com person == -1 (far distance)
ds_test = ds_test.filter(lambda x: x["person"] >= 0)

# Converte para lista en memória
images_list = []
labels_list = []
heights_list = []
widths_list = []

for example in ds_test:
    img = example["image"].numpy()
    label = int(example["person"].numpy())
    h, w = img.shape[:2]
    
    images_list.append(img)
    labels_list.append(label)
    heights_list.append(h)
    widths_list.append(w)

images_array = np.array(images_list, dtype=object)  # dtype=object para permitir tamanhos variáveis
labels_array = np.array(labels_list, dtype="int64")
heights_array = np.array(heights_list, dtype="int32")
widths_array = np.array(widths_list, dtype="int32")

print(f"✓ Dataset carregado: {len(images_array)} imagens")
print(f"  Classes: {np.unique(labels_array)}")
print(f"  Distribuição: no_person={np.sum(labels_array==0)}, person={np.sum(labels_array==1)}")
print(f"  Tamanhos: min={heights_array.min()}x{widths_array.min()}, max={heights_array.max()}x{widths_array.max()}")

## Seção 2 — Estratified Sampling (500 imagens)

In [ ]:
# Amostragem estratificada para balancear classes
indices = np.arange(len(labels_array))

_, sampled_indices = train_test_split(
    indices,
    test_size=SAMPLE_SIZE,
    stratify=labels_array,
    random_state=SEED,
)

# Extrai amostra
sampled_images = images_array[sampled_indices]
sampled_labels = labels_array[sampled_indices]
sampled_heights = heights_array[sampled_indices]
sampled_widths = widths_array[sampled_indices]

print(f"✓ Amostra: {SAMPLE_SIZE} imagens")
print(f"  Distribuição: no_person={np.sum(sampled_labels==0)}, person={np.sum(sampled_labels==1)}")
print(f"  Proporção: {np.sum(sampled_labels==0)/SAMPLE_SIZE*100:.1f}% / {np.sum(sampled_labels==1)/SAMPLE_SIZE*100:.1f}%")

## Seção 3 — Exportar NPZ (uint8 compactado)

In [ ]:
# Converte para uint8 e salva como NPZ
print("Preparando imagens para NPZ (uint8, dtype=object)...")

uint8_images = np.array(
    [np.uint8(img) if img.dtype != np.uint8 else img for img in sampled_images],
    dtype=object
)

npz_path = EXPORT_DIR / "wakevision_test_uint8.npz"
print(f"Salvando em {npz_path}...")

np.savez_compressed(
    npz_path,
    images=uint8_images,
    labels=sampled_labels,
    heights=sampled_heights,
    widths=sampled_widths,
)

file_size_mb = npz_path.stat().st_size / (1024 * 1024)
print(f"✓ NPZ exportado: {file_size_mb:.2f} MB")
print(f"  Caminho: {npz_path}")

## Seção 4 — Exportar PNG + CSV

In [ ]:
# Cria diretório
png_dir = EXPORT_DIR / "images_png"
png_dir.mkdir(parents=True, exist_ok=True)

print(f"Exportando {SAMPLE_SIZE} imagens PNG...")

# Salva imagens
csv_data = []
for i, (img, label, h, w) in enumerate(zip(sampled_images, sampled_labels, sampled_heights, sampled_widths)):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{SAMPLE_SIZE}", end="\r")
    
    # Converte para uint8 se necessário
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    
    # Salva PNG
    img_pil = Image.fromarray(img)
    png_filename = f"image_{i:05d}.png"
    img_pil.save(png_dir / png_filename)
    
    # Registra metadados
    csv_data.append({
        "image_id": i,
        "filename": png_filename,
        "label": label,
        "label_name": "person" if label == 1 else "no_person",
        "original_height": int(h),
        "original_width": int(w),
    })

print(f"✓ {SAMPLE_SIZE} PNGs exportados")

# Salva CSV
csv_path = EXPORT_DIR / "metadata.csv"
df = pd.DataFrame(csv_data)
df.to_csv(csv_path, index=False)
print(f"✓ Metadados salvos: {csv_path}")

## Seção 5 — Carregar Modelos .keras

In [ ]:
from tensorflow.keras import layers

# Registra camada customizada para MCUNet
@tf.keras.utils.register_keras_serializable(package="MCUNet")
class ImageNetNormalization(layers.Layer):
    """Normalização ImageNet: (x/255 - mean) / std"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        self.std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

    def call(self, x):
        return (x / 255.0 - self.mean) / self.std

    def get_config(self):
        return super().get_config()

# Define modelos
MODEL_SPECS = [
    ("MCUNet", MODELS_DIR / "MCUNet_WakeVision.keras"),
    ("MobileNetV3Small", MODELS_DIR / "mobilenetv3_small_wakevision.keras"),
    ("EfficientNetB0", MODELS_DIR / "EfficientNetB0_WakeVision.keras"),
]

models = []
for name, path in MODEL_SPECS:
    if path.exists():
        print(f"✓ Encontrado: {name} <- {path.name}")
        models.append((name, path))
    else:
        print(f"⚠ Não encontrado: {name}")

print(f"\n{len(models)} modelos disponíveis")

## Seção 6 — Converter para TFLite (sem quantização)

In [ ]:
tflite_dir = EXPORT_DIR / "tflite_models"
tflite_dir.mkdir(parents=True, exist_ok=True)

print("Convertendo modelos para TFLite...\n")

for model_name, model_path in models:
    print(f"- {model_name}")
    
    # Carrega modelo
    keras_model = tf.keras.models.load_model(model_path)
    
    # Converte para TFLite (sem quantização por padrão)
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
    ]
    
    tflite_model = converter.convert()
    
    # Salva
    tflite_filename = model_name.replace(" ", "") + ".tflite"
    tflite_path = tflite_dir / tflite_filename
    tflite_path.write_bytes(tflite_model)
    
    size_mb = tflite_path.stat().st_size / (1024 * 1024)
    print(f"  ✓ {tflite_filename} ({size_mb:.2f} MB)")
    
    # Libera memória
    del keras_model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n✓ TFLite models salvos em: {tflite_dir}")

## Seção 7 — Gerar README_MOBILE.md

In [ ]:
readme_content = """# Wake Vision Mobile Export

Dataset otimizado para avaliação em dispositivos móveis.

## Estrutura dos Arquivos

```
export_wakevision/
├── wakevision_test_uint8.npz        # Dataset comprimido (NumPy)
├── metadata.csv                     # Metadados das imagens
├── images_png/                      # Imagens individuais (PNG)
│   ├── image_00000.png
│   ├── image_00001.png
│   └── ...
└── tflite_models/                   # Modelos TFLite
    ├── MCUNet.tflite
    ├── MobileNetV3Small.tflite
    └── EfficientNetB0.tflite
```

## Dados do Dataset

- **Total de imagens:** 500 (amostra estratificada)
- **Classes:** 2 (no_person, person)
- **Tamanhos variáveis:** imagens originais preservadas
- **Redimensionamento (modelo):** 224×224

## Como Carregar em Python

### 1. NPZ (recomendado para prototipagem)

```python
import numpy as np

# Carrega
data = np.load('wakevision_test_uint8.npz', allow_pickle=True)

images = data['images']          # array de objetos (tamanhos variáveis)
labels = data['labels']          # int64 (0=no_person, 1=person)
heights = data['heights']        # int32 (altura original)
widths = data['widths']          # int32 (largura original)

# Processa uma imagem
img = images[0]                  # uint8, tamanho variável
label = labels[0]                # 0 ou 1
print(f"Imagem 0: {img.shape}, label={label}")
```

### 2. PNG + CSV (para aplicativos móveis)

```python
import pandas as pd
from pathlib import Path
from PIL import Image

# Carrega metadados
df = pd.read_csv('metadata.csv')

# Itera sobre imagens
for _, row in df.iterrows():
    img_path = Path('images_png') / row['filename']
    img = Image.open(img_path)
    label = row['label']  # 0 ou 1
    original_size = (row['original_height'], row['original_width'])
    print(f"{row['filename']}: {label}, original size {original_size}")
```

## Como Usar Modelos TFLite

### Python (TensorFlow Lite)

```python
import tensorflow as tf
import numpy as np
from PIL import Image

# Carrega modelo TFLite
interpreter = tf.lite.Interpreter(model_path='tflite_models/MobileNetV3Small.tflite')
interpreter.allocate_tensors()

# Obtém tensor info
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Carrega e processa imagem
img = Image.open('images_png/image_00000.png')
img_resized = img.resize((224, 224))
img_array = np.array(img_resized, dtype=np.float32)

# Normaliza (cada modelo pode ter diferentes escalas)
img_array = img_array / 255.0

# Prepara input
interpreter.set_tensor(input_details[0]['index'], [img_array])

# Executa
interpreter.invoke()

# Obtém predição
output = interpreter.get_tensor(output_details[0]['index'])
class_id = np.argmax(output)
confidence = output[0, class_id]

print(f"Classe: {['no_person', 'person'][class_id]}, confiança: {confidence:.4f}")
```

### Android (Kotlin/Java)

```kotlin
import org.tensorflow.lite.Interpreter
import java.nio.MappedByteBuffer
import java.nio.channels.FileChannel
import java.io.FileInputStream

// Carrega modelo
val modelFile = loadModelFile("MobileNetV3Small.tflite")
val interpreter = Interpreter(modelFile)

// Prepara input (float32, shape 1x224x224x3)
val input = Array(1) { Array(224) { Array(224) { FloatArray(3) } } }

// Prepara output (float32, shape 1x2)
val output = Array(1) { FloatArray(2) { 0f } }

// Executa
interpreter.run(input, output)

// Obtém resultado
val classId = if (output[0][0] > output[0][1]) 0 else 1
val confidence = maxOf(output[0][0], output[0][1])

Log.d("WakeVision", "Class: ${if (classId == 0) "no_person" else "person"}, Conf: $confidence")
```

## Informações Importantes

### Tamanhos de Imagem Variáveis

As imagens do Wake Vision têm **tamanhos originais variáveis**. O arquivo NPZ preserva isso com `dtype=object`.
Para inferência em modelo, sempre redimensione para 224×224.

```python
# ❌ Errado: tentar usar imagem com tamanho variável direto
img = images[0]  # pode ser 180x240, 300x400, etc.
input_tensor = tf.expand_dims(tf.cast(img, tf.float32), 0)  # Erro!

# ✓ Certo: redimensionar antes
img = images[0]
img_resized = tf.image.resize(tf.cast(img, tf.float32), (224, 224))
input_tensor = tf.expand_dims(img_resized, 0)
```

### Normalização

Os modelos esperam entrada float32 em range [0,1] ou [-1,1] dependendo do modelo:

- **EfficientNetB0**: [-1, 1] (ImageNet normalization)
- **MobileNetV3Small**: [0, 1] (rescaling interno)
- **MCUNet**: [-1, 1] (ImageNet normalization)

Quando em dúvida, normalize como: `img_float = img_uint8 / 255.0`

## Avaliação de Performance

Para medir latência e acurácia em dispositivos:

```python
import time
import numpy as np

# Carrega dataset
data = np.load('wakevision_test_uint8.npz', allow_pickle=True)
images, labels = data['images'], data['labels']

# Avalia modelo
correct = 0
times = []

for img, label in zip(images, labels):
    img_resized = tf.image.resize(tf.cast(img, tf.float32), (224, 224))
    img_normalized = img_resized / 255.0
    
    t0 = time.time()
    logits = model(tf.expand_dims(img_normalized, 0))
    pred = tf.argmax(logits[0]).numpy()
    t1 = time.time()
    
    times.append((t1 - t0) * 1000)  # ms
    if pred == label:
        correct += 1

print(f"Acurácia: {correct/len(labels)*100:.2f}%")
print(f"Latência: {np.mean(times):.2f}ms ± {np.std(times):.2f}ms")
```
"""

readme_path = EXPORT_DIR / "README_MOBILE.md"
readme_path.write_text(readme_content)

print(f"✓ README gerado: {readme_path}")

## Seção 8 — Resumo e Estatísticas

In [ ]:
# Resumo da exportação
print("\n" + "="*70)
print("RESUMO DA EXPORTAÇÃO")
print("="*70)

print("\n📊 Dataset:")
print(f"  - Total de imagens: {SAMPLE_SIZE}")
print(f"  - Classe 0 (no_person): {np.sum(sampled_labels==0)} ({np.sum(sampled_labels==0)/SAMPLE_SIZE*100:.1f}%)")
print(f"  - Classe 1 (person): {np.sum(sampled_labels==1)} ({np.sum(sampled_labels==1)/SAMPLE_SIZE*100:.1f}%)")

print("\n📁 Arquivos exportados:")

exported_files = [
    ("NPZ Dataset", EXPORT_DIR / "wakevision_test_uint8.npz"),
    ("Metadados", EXPORT_DIR / "metadata.csv"),
    ("README", EXPORT_DIR / "README_MOBILE.md"),
]

total_size = 0

for label, path in exported_files:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        total_size += path.stat().st_size
        print(f"  ✓ {label:<20} {size_mb:>8.2f} MB  {path.name}")

print(f"\n📂 Diretórios:")
print(f"  ✓ images_png/        {len(list((EXPORT_DIR / 'images_png').glob('*.png'))):<4} imagens")

png_size = sum(f.stat().st_size for f in (EXPORT_DIR / 'images_png').glob('*.png')) / (1024 * 1024)
total_size += png_size * (1024 * 1024)
print(f"                       {png_size:>8.2f} MB")

tflite_files = list((EXPORT_DIR / 'tflite_models').glob('*.tflite'))
if tflite_files:
    tflite_size = sum(f.stat().st_size for f in tflite_files) / (1024 * 1024)
    total_size += tflite_size * (1024 * 1024)
    print(f"  ✓ tflite_models/     {len(tflite_files):<4} modelos")
    print(f"                       {tflite_size:>8.2f} MB")

print(f"\n💾 Tamanho total: {total_size / (1024*1024):.2f} MB")
print(f"\n📍 Diretório de exportação:")
print(f"   {EXPORT_DIR}")

print("\n" + "="*70)
print("✓ Exportação concluída com sucesso!")
print("="*70)